# Proyecto final: Inferencia con Moonshine
## Transformer encoder–decoder para reconocimiento de voz (voz → texto)

**Asignatura:** Procesamiento de Datos Secuenciales con Deep Learning
**Estudiante:** Heidy Romero Ayos
**Profesora:** Nataly Velandia

**Artículo base:** N. Jeffries, E. King, M. Kudlur, G. Nicholson, J. Wang y P. Warden, *"Moonshine: Speech Recognition for Live Transcription and Voice Commands"*, arXiv:2410.15608, 2024.
**Repositorio original:** https://github.com/usefulsensors/moonshine
**Pesos preentrenados:** https://huggingface.co/UsefulSensors/moonshine-base

---

**Objetivo del notebook:** cargar el modelo preentrenado, ejecutar la inferencia sobre audios, y analizar por dentro la arquitectura: entradas y salidas, tensores Q, K y V, y mapas de atención (self-attention del encoder, self-attention enmascarada del decoder y cross-attention).

> Recomendado: en Colab, **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**. Moonshine es liviano y también funciona en CPU.

## 1. Instalación de librerías
- `transformers`: trae la implementación oficial de Moonshine (disponible desde la versión 4.48).
- `librosa`: para cargar y remuestrear audio.
- `jiwer`: para calcular la métrica WER (Word Error Rate).
- `datasets`: para descargar audios de prueba de LibriSpeech.

In [ ]:
!pip install -q "transformers>=4.48" librosa jiwer datasets

In [ ]:
import time
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from transformers import AutoProcessor, MoonshineForConditionalGeneration
import transformers

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Versión de transformers:", transformers.__version__)
print("Dispositivo:", device)

## 2. Carga del modelo y del procesador

- **Procesador (`processor`)**: prepara la entrada (el audio crudo) y convierte la salida (números) en texto. Tiene dos partes: el *feature extractor* (audio → tensor) y el *tokenizer* (tokens ↔ texto).
- **Modelo (`MoonshineForConditionalGeneration`)**: el Transformer encoder–decoder con sus pesos preentrenados.

Usamos `attn_implementation="eager"` para que el modelo **devuelva las matrices de atención**, que necesitamos visualizar.

In [ ]:
MODELO = "UsefulSensors/moonshine-base"   # también existe "UsefulSensors/moonshine-tiny" (más pequeño)

processor = AutoProcessor.from_pretrained(MODELO)
model = MoonshineForConditionalGeneration.from_pretrained(
    MODELO,
    attn_implementation="eager"   # necesario para obtener los pesos de atención
).to(device)
model.eval()

n_param = sum(p.numel() for p in model.parameters())
print(f"Parámetros del modelo: {n_param/1e6:.1f} millones")

In [ ]:
# Hiperparámetros principales de la arquitectura (vienen en la configuración del modelo)
cfg = model.config
for clave in ["hidden_size", "intermediate_size", "encoder_num_hidden_layers",
              "decoder_num_hidden_layers", "encoder_num_attention_heads",
              "decoder_num_attention_heads", "vocab_size", "partial_rotary_factor",
              "rope_theta", "max_position_embeddings"]:
    if hasattr(cfg, clave):
        print(f"{clave:30s}: {getattr(cfg, clave)}")

## 3. Datos de entrada: el audio

La entrada de Moonshine es **audio crudo** (la forma de onda), muestreado a **16.000 Hz**: un vector de números donde cada número es la amplitud del sonido en un instante.

A diferencia de Whisper, que primero convierte el audio en un espectrograma de 30 segundos, **Moonshine recibe directamente la forma de onda** y la procesa con capas convolucionales.

Usamos audios de prueba del dataset **LibriSpeech** (lecturas de audiolibros en inglés), que traen su transcripción real para poder comparar.

In [ ]:
def a_numpy(muestra_audio):
    """Convierte el audio de una muestra de 'datasets' en (array, sampling_rate).
    Funciona con versiones antiguas (dict) y nuevas (AudioDecoder) de la librería."""
    if isinstance(muestra_audio, dict):
        return np.asarray(muestra_audio["array"], dtype=np.float32), muestra_audio["sampling_rate"]
    datos = muestra_audio.get_all_samples()
    return datos.data.mean(dim=0).numpy().astype(np.float32), datos.sample_rate

ds = None
try:
    from datasets import load_dataset
    ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
    audio, sr = a_numpy(ds[0]["audio"])
    referencia = ds[0]["text"]
    print("Audio cargado desde LibriSpeech (dummy).")
except Exception as e:
    # Plan B: un fragmento de LibriSpeech que trae librosa
    print("No se pudo usar 'datasets' (", type(e).__name__, "). Uso el ejemplo de librosa.")
    audio, sr = librosa.load(librosa.ex("libri1"), sr=16000)
    referencia = None

# Moonshine trabaja a 16 kHz: si el audio viene con otra frecuencia, lo remuestreamos
if sr != 16000:
    audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    sr = 16000

duracion = len(audio) / sr
print(f"Muestras: {len(audio)} | Frecuencia: {sr} Hz | Duración: {duracion:.2f} s")
print("Transcripción real:", referencia)
display(Audio(audio, rate=sr))

In [ ]:
# Visualización de la forma de onda (la secuencia de entrada)
t = np.arange(len(audio)) / sr
plt.figure(figsize=(12, 3))
plt.plot(t, audio, linewidth=0.5)
plt.title("Forma de onda del audio de entrada")
plt.xlabel("Tiempo (s)")
plt.ylabel("Amplitud")
plt.tight_layout()
plt.show()

## 4. Preprocesamiento

El procesador convierte el audio en el tensor `input_values` con forma **(batch, número de muestras)**. No hay relleno a 30 segundos: el tensor mide exactamente lo que dura el audio. Esta es una de las innovaciones de Moonshine frente a Whisper.

In [ ]:
inputs = processor(audio, sampling_rate=sr, return_tensors="pt").to(device)

for nombre, tensor in inputs.items():
    print(f"{nombre:15s}: forma {tuple(tensor.shape)}  tipo {tensor.dtype}")

## 5. Inferencia: generación del texto

El decoder genera la transcripción **token por token** (de forma autorregresiva): predice un token, lo agrega a la secuencia y vuelve a predecir, hasta generar el token de fin.

La documentación recomienda limitar la longitud máxima según la duración del audio (unos **6,5 tokens por segundo**), para evitar que el modelo entre en repeticiones.

In [ ]:
def transcribir(audio_np, sr=16000):
    """Recibe un audio (numpy, 16 kHz) y devuelve (texto, ids generados, tiempo en segundos)."""
    entradas = processor(audio_np, sampling_rate=sr, return_tensors="pt").to(device)
    max_tokens = max(10, int(len(audio_np) / sr * 6.5))   # límite: ~6,5 tokens por segundo
    inicio = time.time()
    with torch.no_grad():
        ids = model.generate(**entradas, max_length=max_tokens)
    tiempo = time.time() - inicio
    texto = processor.decode(ids[0], skip_special_tokens=True)
    return texto, ids, tiempo

# Calentamiento: la primera inferencia siempre es más lenta (se prepara la GPU y el modelo).
# La hacemos una vez sin medir, para que el tiempo reportado sea el real.
_ = transcribir(audio, sr)

texto, generated_ids, tiempo = transcribir(audio, sr)

print("Transcripción del modelo:", texto)
print("Transcripción real      :", referencia)
print(f"Tiempo de inferencia: {tiempo:.3f} s para {duracion:.2f} s de audio")
print(f"Factor de tiempo real (RTF): {tiempo/duracion:.3f}  (menor que 1 = más rápido que el tiempo real)")

In [ ]:
# La salida del decoder es una secuencia de ids (tokens). Así se ve antes de convertirla en texto:
tokens = processor.tokenizer.convert_ids_to_tokens(generated_ids[0])
print("Ids generados:", generated_ids[0].tolist())
print("Tokens       :", tokens)

## 6. Arquitectura del modelo

Moonshine sigue el esquema **encoder–decoder** del Transformer original, con cambios importantes:

| Parte | Qué hace |
|---|---|
| **Entrada** | Audio crudo a 16 kHz, **sin espectrograma mel** |
| **Stem convolucional** (el "embedding" del audio) | 3 convoluciones 1D: kernel 127 / stride 64 → tanh; kernel 7 / stride 3 → GELU; kernel 3 / stride 2 → GELU. Comprimen el audio **384×** (≈ 41,7 vectores por segundo). Es análogo al *patch embedding* del ViT |
| **Encoder** | Bloques Transformer con **self-attention** + **RoPE** + red feed-forward con **GELU** |
| **Decoder** | Bloques Transformer con **self-attention enmascarada** + RoPE, **cross-attention** al encoder y red feed-forward con **SwiGLU** |
| **Tokenizador** | BPE de LLaMA: 32.000 tokens + 768 especiales (por eso las palabras se parten en subpalabras) |
| **Salida** | Capa lineal + softmax sobre el vocabulario → siguiente token (generación autorregresiva) |

**Tamaños según el artículo (Tabla 1):**

| | Moonshine Tiny | Whisper tiny.en | **Moonshine Base** | Whisper base.en |
|---|---|---|---|---|
| Dimensión | 288 | 384 | **416** | 512 |
| Capas encoder / decoder | 6 / 6 | 4 / 4 | **8 / 8** | 6 / 6 |
| Cabezas de atención | 8 | 6 | **8** | 8 |
| Parámetros | 27,1 M | 37,8 M | **61,5 M** | 72,6 M |

Entrenamiento (según el artículo): ~200.000 horas de audio en inglés, 32 GPUs H100, 250.000 pasos.

Imprimimos la estructura real del modelo:

In [ ]:
print(model)

## 7. Creación de los tensores Q, K y V

En cada capa de atención, los tensores se obtienen multiplicando la entrada por matrices de pesos aprendidas:

$$Q = X W_Q \qquad K = X W_K \qquad V = X W_V$$

En el código, esas matrices son las capas lineales `q_proj`, `k_proj` y `v_proj`. La diferencia clave está en **de dónde viene X**:

| Tipo de atención | Q viene de… | K y V vienen de… |
|---|---|---|
| Self-attention del **encoder** | el audio | el audio |
| Self-attention **enmascarada del decoder** | los tokens generados | los tokens generados |
| **Cross-attention** | los tokens generados (decoder) | la salida del **encoder** (audio) |

Para verlo, ponemos *hooks* (ganchos) que capturan la salida de `q_proj`, `k_proj` y `v_proj` en la **primera capa** del encoder y del decoder, durante una pasada completa del modelo.

In [ ]:
capturas = {}
ganchos = []

def crear_gancho(nombre):
    def gancho(modulo, entrada, salida):
        capturas[nombre] = salida.detach()
    return gancho

# Registramos ganchos en q_proj, k_proj y v_proj de la primera capa (layers.0) del encoder y del decoder
for nombre, modulo in model.named_modules():
    if ".layers.0." in nombre and nombre.split(".")[-1] in ("q_proj", "k_proj", "v_proj"):
        ganchos.append(modulo.register_forward_hook(crear_gancho(nombre)))

# Una pasada completa (teacher forcing) con los tokens ya generados, pidiendo también las atenciones
with torch.no_grad():
    salida = model(**inputs, decoder_input_ids=generated_ids,
                   output_attentions=True, use_cache=False)

for g in ganchos:
    g.remove()

n_audio = salida.encoder_last_hidden_state.shape[1]
n_tokens = generated_ids.shape[1]
print(f"Vectores de audio a la salida del encoder: {n_audio}  (~{n_audio/duracion:.0f} por segundo)")
print(f"Tokens en el decoder: {n_tokens}\n")
for nombre, tensor in capturas.items():
    print(f"{nombre:55s} forma {tuple(tensor.shape)}")

**Cómo leer las formas** `(batch, longitud, dimensión)`:
- En el **encoder**, Q, K y V tienen la longitud de la secuencia de **audio**.
- En la **self-attention del decoder** (`self_attn`), Q, K y V tienen la longitud de los **tokens**.
- En la **cross-attention** (`encoder_attn`), **Q tiene la longitud de los tokens** (viene del decoder), pero **K y V tienen la longitud del audio** (vienen del encoder). Así cada token "pregunta" qué parte del audio le sirve.

La dimensión final es `número de cabezas × dimensión por cabeza`: la atención es **multi-head**.

## 8. Análisis del mecanismo de atención

La atención se calcula con **Scaled Dot-Product Attention**:

$$\text{Atención}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

El modelo devolvió tres grupos de matrices de atención (una por capa), cada una con forma `(batch, cabezas, consultas, claves)`:

In [ ]:
print("Capas del encoder (self-attention):     ", len(salida.encoder_attentions), "| forma:", tuple(salida.encoder_attentions[0].shape))
print("Capas del decoder (self-attention):     ", len(salida.decoder_attentions), "| forma:", tuple(salida.decoder_attentions[0].shape))
print("Capas del decoder (cross-attention):    ", len(salida.cross_attentions),   "| forma:", tuple(salida.cross_attentions[0].shape))

In [ ]:
def limpiar(tok):
    return tok.replace("▁", " ").strip() or tok

etiquetas = [limpiar(t) for t in tokens]
seg_por_vector = duracion / n_audio

# --- 8.1 Self-attention enmascarada del decoder (promedio de cabezas, última capa) ---
att_dec = salida.decoder_attentions[-1][0].mean(dim=0).cpu().numpy()
plt.figure(figsize=(7, 6))
plt.imshow(att_dec, cmap="Blues")
plt.xticks(range(n_tokens), etiquetas, rotation=90)
plt.yticks(range(n_tokens), etiquetas)
plt.title("Decoder: self-attention enmascarada (última capa)")
plt.xlabel("Token al que presta atención (Key)")
plt.ylabel("Token que consulta (Query)")
plt.colorbar(label="Peso de atención")
plt.tight_layout()
plt.show()

La matriz es **triangular inferior**: cada token solo presta atención a sí mismo y a los anteriores, nunca a los futuros. Es la **máscara causal** que vimos en la clase del Transformer decoder.

In [ ]:
# --- 8.2 Cross-attention: qué parte del audio mira cada token (promedio de cabezas, última capa) ---
att_cross = salida.cross_attentions[-1][0].mean(dim=0).cpu().numpy()

plt.figure(figsize=(12, 6))
plt.imshow(att_cross, aspect="auto", cmap="magma",
           extent=[0, duracion, n_tokens - 0.5, -0.5])
plt.yticks(range(n_tokens), etiquetas)
plt.xlabel("Tiempo del audio (s)")
plt.ylabel("Token generado")
plt.title("Cross-attention: cada token consulta el audio (última capa)")
plt.colorbar(label="Peso de atención")
plt.tight_layout()
plt.show()

En la cross-attention se espera ver una **diagonal**: a medida que el modelo escribe palabras, su atención avanza en el tiempo del audio. Es el "alineamiento" entre lo que se dice y lo que se escribe. Las capas de la mitad del decoder suelen mostrar esta diagonal más clara; puedes cambiar `-1` por otra capa para comparar.

In [ ]:
# --- 8.3 Cross-attention por capa: comparar cómo cambia el alineamiento ---
n_capas = len(salida.cross_attentions)
fig, ejes = plt.subplots(2, (n_capas + 1) // 2, figsize=(16, 6))
for i, ax in enumerate(ejes.flat):
    if i >= n_capas:
        ax.axis("off"); continue
    mapa = salida.cross_attentions[i][0].mean(dim=0).cpu().numpy()
    ax.imshow(mapa, aspect="auto", cmap="magma")
    ax.set_title(f"Capa {i+1}")
    ax.set_xlabel("audio"); ax.set_ylabel("tokens")
plt.suptitle("Cross-attention en cada capa del decoder")
plt.tight_layout()
plt.show()

In [ ]:
# --- 8.4 Self-attention del encoder (promedio de cabezas, primera y última capa) ---
fig, ejes = plt.subplots(1, 2, figsize=(13, 5))
for ax, capa in zip(ejes, [0, -1]):
    mapa = salida.encoder_attentions[capa][0].mean(dim=0).cpu().numpy()
    im = ax.imshow(mapa, cmap="viridis", extent=[0, duracion, duracion, 0])
    ax.set_title(f"Encoder: self-attention, capa {'1' if capa == 0 else 'final'}")
    ax.set_xlabel("Tiempo (s) — Key")
    ax.set_ylabel("Tiempo (s) — Query")
plt.colorbar(im, ax=ejes, label="Peso de atención")
plt.show()

En el encoder, cada fragmento de audio presta atención sobre todo a sus **vecinos cercanos en el tiempo** (la franja de la diagonal), y algunas cabezas miran zonas más lejanas para captar contexto. Como usa **RoPE**, la atención depende de la **distancia relativa** entre fragmentos.

## 9. Prueba con un audio propio

Sube un archivo de audio en **inglés** (`.wav`, `.mp3`, `.m4a`…), idealmente de menos de 30 segundos. Puedes grabarte con el celular.

In [ ]:
try:
    from google.colab import files
    subidos = files.upload()
    archivo = list(subidos.keys())[0]
    audio_propio, _ = librosa.load(archivo, sr=16000)   # carga y remuestrea a 16 kHz
    display(Audio(audio_propio, rate=16000))
    texto_propio, _, t_propio = transcribir(audio_propio)
    print("Transcripción:", texto_propio)
    print(f"Tiempo: {t_propio:.3f} s para {len(audio_propio)/16000:.2f} s de audio")
except Exception as e:
    print("No se cargó audio propio:", e)

## 10. Evaluación: métricas de desempeño

- **WER (Word Error Rate):** porcentaje de palabras mal transcritas (sustituidas, eliminadas o insertadas). **Menor es mejor.**
- **RTF (Real-Time Factor):** tiempo de inferencia ÷ duración del audio. Menor que 1 significa que transcribe más rápido que el tiempo real.

Antes de comparar, normalizamos los textos (minúsculas, sin signos de puntuación y con abreviaturas escritas completas, como "Mr." → "mister"), porque LibriSpeech está escrito en mayúsculas, sin puntuación y sin abreviaturas. Si no se normaliza, el WER contaría como error diferencias que solo son de formato.

In [ ]:
import re
import jiwer

# Abreviaturas que el modelo escribe abreviadas pero LibriSpeech escribe completas
ABREVIATURAS = {r"\bmr\b": "mister", r"\bmrs\b": "missus", r"\bdr\b": "doctor", r"\bst\b": "saint"}

def normalizar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9' ]", " ", texto)
    for patron, completo in ABREVIATURAS.items():
        texto = re.sub(patron, completo, texto)
    return re.sub(r"\s+", " ", texto).strip()

if ds is not None:
    N = min(30, len(ds))
    refs, hyps, tiempos, duraciones = [], [], [], []
    for i in range(N):
        a, s = a_numpy(ds[i]["audio"])
        if s != 16000:
            a = librosa.resample(a, orig_sr=s, target_sr=16000)
        hip, _, t_inf = transcribir(a)
        refs.append(normalizar(ds[i]["text"]))
        hyps.append(normalizar(hip))
        tiempos.append(t_inf)
        duraciones.append(len(a) / 16000)

    wer = jiwer.wer(refs, hyps)
    rtf = sum(tiempos) / sum(duraciones)
    print(f"Audios evaluados: {N}")
    print(f"WER: {wer*100:.2f} %")
    print(f"RTF promedio: {rtf:.3f}  ({device})")

    print("\nEjemplos (referencia vs. modelo):")
    for r, h in list(zip(refs, hyps))[:3]:
        print("REF:", r)
        print("HIP:", h, "\n")
else:
    print("No hay dataset disponible para calcular el WER.")

In [ ]:
# Error por audio: ¿en qué audios se equivoca más?
if ds is not None:
    wer_por_audio = [jiwer.wer(r, h) for r, h in zip(refs, hyps)]
    plt.figure(figsize=(12, 3.5))
    plt.bar(range(1, N + 1), [w * 100 for w in wer_por_audio])
    plt.axhline(wer * 100, color="red", linestyle="--", label=f"WER global {wer*100:.1f} %")
    plt.xlabel("Audio")
    plt.ylabel("WER (%)")
    plt.title("WER por audio")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 11. Métricas del artículo vs. nuestros resultados

El artículo evalúa con **WER** en los 8 datasets del *Open ASR Leaderboard*, usando decodificación *greedy* y un límite de 6 tokens por segundo de audio (Tablas 2 y 3 del artículo):

| Dataset | Whisper base.en | **Moonshine Base** | Whisper tiny.en | Moonshine Tiny |
|---|---|---|---|---|
| AMI | 21,13 | **17,79** | 24,24 | 22,77 |
| Earnings22 | 15,09 | 17,65 | 19,12 | 21,25 |
| GigaSpeech | 12,83 | **12,19** | 14,08 | 14,41 |
| LibriSpeech (clean) | 4,25 | **3,23** | 5,66 | 4,52 |
| LibriSpeech (other) | 10,35 | **8,18** | 15,45 | 11,71 |
| SPGISpeech | 4,26 | 5,46 | 5,93 | 7,70 |
| TED-LIUM | 4,87 | 5,22 | 5,97 | 5,64 |
| VoxPopuli | 9,76 | 10,81 | 12,00 | 13,27 |
| **Promedio** | 10,32 | **10,07** | 12,81 | 12,66 |

Además, el artículo reporta que Moonshine Tiny necesita **5 veces menos cómputo** que Whisper tiny.en para un audio de 10 segundos, y que Moonshine ofrece hasta **3 veces menos latencia**.

In [ ]:
# Gráfica comparativa del artículo: Moonshine Base vs. Whisper base.en
datasets_art = ["AMI", "Earnings22", "GigaSpeech", "LS clean", "LS other", "SPGISpeech", "TED-LIUM", "VoxPopuli"]
whisper_base = [21.13, 15.09, 12.83, 4.25, 10.35, 4.26, 4.87, 9.76]
moon_base    = [17.79, 17.65, 12.19, 3.23,  8.18, 5.46, 5.22, 10.81]

x = np.arange(len(datasets_art))
plt.figure(figsize=(12, 4))
plt.bar(x - 0.2, whisper_base, width=0.4, label="Whisper base.en (72,6 M)")
plt.bar(x + 0.2, moon_base, width=0.4, label="Moonshine Base (61,5 M)")
plt.xticks(x, datasets_art)
plt.ylabel("WER (%)  — menor es mejor")
plt.title("WER reportado en el artículo (Tabla 2)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Nuestro resultado frente al del artículo en LibriSpeech
wer_articulo = 3.23   # Moonshine Base, LibriSpeech test-clean (2.620 audios)
if ds is not None:
    print(f"WER del artículo (LibriSpeech test-clean, 2.620 audios): {wer_articulo:.2f} %")
    print(f"WER de nuestra prueba (LibriSpeech, {N} audios):         {wer*100:.2f} %")
    print(f"RTF de nuestra prueba en {device}:                        {rtf:.3f}  (≈ {1/rtf:.0f} veces más rápido que el tiempo real)")

**Cómo interpretar la diferencia:**
- Nuestra muestra es pequeña (30 audios de otra partición de LibriSpeech), así que un solo error cambia mucho el porcentaje.
- La normalización del texto influye: sin convertir "Mr." en "mister", el WER sube por diferencias que solo son de formato.
- Los errores reales que quedan son principalmente **acústicos**, por ejemplo "roast beef" → "rose beef": dos palabras que suenan casi igual.

## 12. Notas para el análisis (README y sustentación)

**Innovaciones de Moonshine frente a Whisper (según el artículo):**
1. **Entrada sin relleno:** Whisper rellena todo audio con ceros hasta 30 s, así que gasta lo mismo en 2 s que en 30 s. Moonshine procesa solo la duración real del audio.
2. **RoPE en vez de posiciones absolutas:** el artículo muestra que al quitarle el relleno a Whisper, su WER sube de **5,21 % a 107,38 %**, porque sus posiciones absolutas solo funcionan con 30 s exactos. RoPE aplica la posición rotando Q y K, y la atención depende de la distancia relativa.
3. **Stem convolucional sobre audio crudo:** no necesita calcular un espectrograma mel.
4. **Modelos pequeños (Tiny 27,1 M y Base 61,5 M):** pensados para transcripción en vivo y dispositivos de pocos recursos.

**Limitaciones (del artículo y de nuestras pruebas):**
- Solo transcribe **inglés**.
- **Audios muy cortos (< 1 s):** tiende a repetir tokens; por eso le va peor en Earnings22.
- **Audios largos (> 30 s):** aumentan las alucinaciones.
- **Audio muy bajito** (por debajo de −40 dB): Whisper es más robusto.
- Errores por **similitud acústica** cuando falta contexto ("roast" → "rose").